In [ ]:
# =========================
# Data Splitting in Python
# =========================
from sklearn.model_selection import train_test_split

# Assume `df` is your DataFrame and 'target' is the column to predict
X = df.drop('target', axis=1)
y = df['target']

# 1) Simple Train/Test Split (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,  # sets seed for reproducibility
    shuffle=True      # set to False for time series data
)

# 2) Three-way Split: Train (60%), Val (20%), Test (20%) - First split off Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Then split remaining into Train and Val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,  # 0.25 * 0.80 = 0.20
    random_state=42,
    shuffle=True
)

# 3) For classification: preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [ ]:
# ==========================================
# Python: Rolling vs Expanding Window CV
# ==========================================
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# Assume X, y are numpy arrays or pandas DataFrame/Series
# n_splits = number of folds, test_size = h, max_train_size=None (default)

# ----- Expanding Window CV -----
exp_cv = TimeSeriesSplit(n_splits=5, test_size=10, max_train_size=None)
for train_idx, test_idx in exp_cv.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

# ----- Rolling Window CV -----
# window_size = W, test_size = h
W, h = 50, 10
roll_cv = TimeSeriesSplit(n_splits=5,
                          test_size=h,
                          max_train_size=W)
for train_idx, test_idx in roll_cv.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

In [ ]:
# ==========================================
# Python: Rolling-Window CV Example
# ==========================================
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Assume X, y are numpy arrays or pandas DataFrame/Series of length N
W = 50               # fixed training window size
h = 10               # test horizon per fold
n_splits = 5         # number of folds

# Rolling-window CV: train on last W points, test on next h, step by h
roll_cv = TimeSeriesSplit(
    n_splits=n_splits,
    test_size=h,
    max_train_size=W
)

model = LinearRegression()
mses = []

for train_idx, test_idx in roll_cv.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mses.append(mean_squared_error(y_test, preds))

# 'mses' now has one MSE per rolling-window fold

In [ ]:
# ==========================================
# Repeated Hold–Out in Python
# ==========================================
import numpy as np
from sklearn.model_selection import ShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Set up repeated hold‐out splits
rs = ShuffleSplit(n_splits=10,
                  test_size=0.2,
                  random_state=42)

errors = []
for train_idx, test_idx in rs.split(X):
    # Partition data
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit model on training split
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict and compute test error
    y_pred = model.predict(X_test)
    err = mean_squared_error(y_test, y_pred)
    errors.append(err)

# Average error over R repetitions
mean_err = np.mean(errors)

In [ ]:
# ==========================================
# Grid Search for Hyperparameters in Python
# ==========================================
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 1) Split data into train / validation / test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# 2) Define grid of candidate alphas
alphas = [0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_err   = np.inf

# 3) Grid search: train on X_train, evaluate on X_val
for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)                  # fit on training set
    preds = model.predict(X_val)                 # predict on validation set
    err   = mean_squared_error(y_val, preds)     # compute validation MSE
    if err < best_err:
        best_err   = err
        best_alpha = a

# 4) Retrain on train + val with best alpha
X_full = np.vstack([X_train, X_val])
y_full = np.hstack([y_train, y_val])
final_model = Ridge(alpha=best_alpha)
final_model.fit(X_full, y_full)                  # fit on combined data

# 5) Evaluate final model on held-out test set
test_preds = final_model.predict(X_test)
test_err   = mean_squared_error(y_test, test_preds)

In [ ]:
# ==========================================
# Nested Cross‐Validation in Python
# ==========================================
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, GridSearchCV

# Set up outer and inner CV
K_out, K_in = 5, 3
outer_cv = KFold(n_splits=K_out, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=K_in, shuffle=True, random_state=42)

# Hyperparameter grid
param_grid = {'alpha': [0.1, 1.0, 10.0]}
outer_errors = []

for train_idx, test_idx in outer_cv.split(X):
    # Split outer train/test
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Inner loop: grid search on X_train
    gs = GridSearchCV(
        estimator=Ridge(),
        param_grid=param_grid,
        cv=inner_cv,
        scoring='neg_mean_squared_error'
    )
    gs.fit(X_train, y_train)                  # tunes alpha on inner folds
    best_model = gs.best_estimator_           # model with lambda*(k)

    # Evaluate on outer test set
    y_pred = best_model.predict(X_test)
    err = mean_squared_error(y_test, y_pred)
    outer_errors.append(err)

# Final nested CV error estimate
nested_mse = np.mean(outer_errors)

In [3]:
# ==========================================
# Nonparametric Bootstrap in Python
# ==========================================
import numpy as np

# 1) Original data
X = np.array([2.3, 4.1, 5.6, 3.8, 4.9, 6.0, 5.2, 4.7, 5.9, 3.3])
n = len(X)

# 2) Statistic to bootstrap (e.g., sample mean)
def stat(x):
    return np.mean(x)

# 3) Draw B bootstrap samples with replacement
B = 1000
theta_star = np.empty(B)
for b in range(B):
    Xb = np.random.choice(X, size=n, replace=True)  # resample indices
    theta_star[b] = stat(Xb)                        # compute statistic

# 4) theta_star now holds the B bootstrap replicates of stat(X)

In [4]:
# ==========================================
# Parametric Bootstrapping in Python
# ==========================================
import numpy as np
from scipy.stats import norm

# 1) Original data
X = np.array([2.3, 4.1, 5.6, 3.8, 4.9, 6.0, 5.2, 4.7, 5.9, 3.3])
n = len(X)

# 2) Fit parametric model (Normal): estimate mu, sigma
mu_hat    = X.mean()
sigma_hat = X.std(ddof=1)

# 3) Statistic to bootstrap (e.g., sample mean)
def stat(x):
    return np.mean(x)

# 4) Draw B parametric bootstrap samples
B = 1000
theta_star = np.empty(B)
for b in range(B):
    Xb = norm.rvs(loc=mu_hat, scale=sigma_hat, size=n)  # sample from N(mu_hat, sigma_hat^2)
    theta_star[b] = stat(Xb)                            # compute statistic

# 5) theta_star now holds B replicates of stat(X) under the fitted model

In [5]:
# ==========================================
# Residual Bootstrapping in Python
# ==========================================
import numpy as np
from sklearn.linear_model import LinearRegression

# 1) Simulate or load data: X (n×p), y (n,)
#    Here we create a simple example:
np.random.seed(0)
n, p = 100, 1
X = np.random.uniform(0, 10, size=(n, p))
beta_true = np.array([2.5])
y = X.dot(beta_true) + np.random.normal(scale=3.0, size=n)

# 2) Fit original model and compute residuals
model = LinearRegression().fit(X, y)
y_pred    = model.predict(X)
residuals = y - y_pred

# 3) Residual bootstrap replicates
B = 500
beta_star = np.empty((B, p))
for b in range(B):
    eps_star = np.random.choice(residuals, size=n, replace=True)
    y_star   = y_pred + eps_star                # new responses
    model_b  = LinearRegression().fit(X, y_star)
    beta_star[b] = model_b.coef_

# beta_star now holds B bootstrap estimates of the regression coefficients

In [6]:

# ==========================================
# Block Bootstrapping in Python (Time Series)
# ==========================================
import numpy as np

# 1) Simulate an example time series X of length n
np.random.seed(0)
n = 100
t = np.linspace(0, 4*np.pi, n)
X = np.sin(t) + np.random.normal(scale=0.3, size=n)

# 2) Define block length l and number of blocks m per replicate
l = 20
m = int(np.ceil(n / l))    # number of blocks to sample

# 3) Precompute overlapping blocks B_j of length l
#    B has shape (n-l+1, l)
B = np.lib.stride_tricks.sliding_window_view(X, window_shape=l)

# 4) Generate B_bootstrap bootstrap replicates
B_reps = 200
bootstrap_replicates = np.empty((B_reps, n))
for b in range(B_reps):
    # sample m block‐indices with replacement
    idx = np.random.randint(0, B.shape[0], size=m)
    # concatenate blocks and truncate to length n
    Xb = np.concatenate(B[idx])[:n]
    bootstrap_replicates[b] = Xb

# bootstrap_replicates is now (B_reps * n), each row a bootstrapped series

In [ ]:
# ==========================================
# Bootstrap Bias & Variance Estimation in Python
# ==========================================
import numpy as np

# theta_star: array of B bootstrap replicates of the statistic
# theta_hat : original statistic computed on X
# Example:
# theta_hat  = stat(X)
# theta_star = np.array([...])  # shape (B,)

# 1) Bootstrap mean
theta_bar = theta_star.mean()            # (1/B) * sum theta_star

# 2) Bias estimate & bias‐corrected
bias_hat  = theta_bar - theta_hat        # bootstrap estimate of bias
theta_corr = 2*theta_hat - theta_bar     # bias‐corrected estimate

# 3) Bootstrap variance & standard error
var_hat   = theta_star.var(ddof=1)       # 1/(B-1)*sum(theta_star - theta_bar)^2
se_hat    = np.sqrt(var_hat)             # standard error

In [ ]:
# ==========================================
# Bootstrap Confidence Intervals in Python
# ==========================================
import numpy as np

# Inputs:
# theta_hat  = original estimate (float)
# theta_star = array of B bootstrap replicates of the estimate, shape (B,)
# se_hat     = original standard error of theta_hat (bootstrap or analytic)
# se_star    = array of B bootstrap SEs for each replicate (via nested bootstrap or analytic)
# alpha      = significance level (e.g., 0.05 for 95% CI)

# 1) Percentile CI
lo_p = np.percentile(theta_star, 100*(alpha/2))
hi_p = np.percentile(theta_star, 100*(1-alpha/2))
ci_percentile = (lo_p, hi_p)

# 2) Basic CI
ci_basic = (2*theta_hat - hi_p,
            2*theta_hat - lo_p)

# 3) Studentized CI
t_star = (theta_star - theta_hat) / se_star
t_lo = np.percentile(t_star, 100*(1-alpha/2))
t_hi = np.percentile(t_star, 100*(alpha/2))
ci_studentized = (theta_hat - t_lo * se_hat,
                  theta_hat - t_hi * se_hat)

In [ ]:
# ==========================================
# Computing AIC & BIC in Python
# ==========================================
import numpy as np
from sklearn.linear_model import LinearRegression

# X: feature matrix (n×p), y: response vector (n,)
model = LinearRegression().fit(X, y)
y_pred = model.predict(X)
residuals = y - y_pred

n = len(y)
p = X.shape[1] + 1        # number of parameters (features + intercept)
rss = np.sum(residuals**2)

# Estimate variance by MLE (divide by n)
sigma2 = rss / n

# Log‐likelihood under Gaussian noise
ll = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)

# AIC and BIC
aic = -2 * ll + 2 * p
bic = -2 * ll + np.log(n) * p

# aic, bic now hold the information criteria values

In [ ]:
# ==========================================
# Best Subset Selection in Python
# ==========================================
import numpy as np
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# X: n*p feature matrix, y: length-n target vector
n, p = X.shape
results = {}  # store best model info for each k

for k in range(1, p+1):
    best_r2 = -np.inf
    best_subset = None
    # enumerate all subsets of size k
    for cols in combinations(range(p), k):
        X_sub = X[:, cols]
        model = LinearRegression().fit(X_sub, y)
        r2 = r2_score(y, model.predict(X_sub))
        if r2 > best_r2:
            best_r2 = r2
            best_subset = cols
    results[k] = {'subset': best_subset, 'r2': best_r2}

# Include null model M0
y_mean = np.full(n, y.mean())
results[0] = {'subset': (), 'r2': r2_score(y, y_mean)}

# Now results[k] holds the best subset and its R2 for each k
# Select k* by evaluating results[k] via AIC/BIC/CV/validation

In [ ]:
# ==========================================
# Stepwise Selection in Python
# ==========================================
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector

# X: n*p feature matrix or DataFrame, y: target vector
estimator = LinearRegression()

# Forward stepwise: add one feature at a time until k features selected
k = 5  # desired number of features
sfs = SequentialFeatureSelector(
    estimator,
    n_features_to_select=k,
    direction='forward',  # set 'backward' for backward
    scoring='r2',        # use R2 to evaluate each candidate step
    cv=5,                # 5-fold internal CV to compare additions
    n_jobs=-1
)

sfs.fit(X, y)

# Boolean mask: True for selected features, False otherwise
mask = sfs.get_support()        # e.g. [False, True, False, True, ...]
# Indices of selected features
selected_indices = np.where(mask)[0].tolist()

# If X is a pandas DataFrame, get column names:
# selected_cols = X.columns[mask].tolist()

# To transform X to keep only selected features:
X_selected = sfs.transform(X)   # returns X[:, mask] or df[selected_cols]

# Additional attributes:
# sfs.ranking_  — 1 for selected features, >1 rank for others
# sfs.n_features_to_select  — the integer k